# Explore LRO PostgreSQL Database

This notebook inspects the project database structure before doing analysis: schemas, tables, columns, keys, indexes, row counts, samples, and basic profiling.

Default connection values come from `README.md` / `docker-compose.yml`:

- Host: `localhost`
- Port: `5433`
- Database: `database`
- User: `admin`
- Password: `password`

If the connection cell fails, start the database first with:

```bash
docker compose up -d --build
```

## 1. Imports and Connection Settings

In [1]:
import os
from IPython.display import display
import pandas as pd

try:
    from sqlalchemy import create_engine, inspect, text
except ImportError as exc:
    raise ImportError(
        "Missing dependencies. Install them with: "
        "pip install pandas sqlalchemy psycopg2-binary notebook"
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "database")
DB_USER = os.getenv("DB_USER", "admin")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

## 2. Test the Database Connection

In [2]:
connection_check = pd.read_sql_query(
    text("""
    SELECT
        current_database() AS database,
        current_user AS user,
        inet_server_addr() AS server_address,
        inet_server_port() AS server_port,
        version() AS postgres_version
    """),
    engine,
)
display(connection_check)

,database,user,server_address,server_port,postgres_version
0,database,admin,172.18.0.2,5432,"PostgreSQL 16.13 (Debian 16.13-1.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, ..."


## 3. Schemas and Tables

In [3]:
schemas = pd.read_sql_query(
    text("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'pg_toast')
    ORDER BY schema_name
    """),
    engine,
)
display(schemas)

,schema_name
0,public


In [4]:
tables = pd.read_sql_query(
    text("""
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
    """),
    engine,
)
display(tables)

,table_schema,table_name,table_type
0,public,datastream,BASE TABLE
1,public,method,BASE TABLE
2,public,owner,BASE TABLE
3,public,processing_level,BASE TABLE
4,public,qualifier,BASE TABLE
5,public,site,BASE TABLE
6,public,staging,BASE TABLE
7,public,unit,BASE TABLE
8,public,variable,BASE TABLE


## 4. Columns and Data Types

In [5]:
columns = pd.read_sql_query(
    text("""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        udt_name,
        character_maximum_length,
        numeric_precision,
        numeric_scale,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name, ordinal_position
    """),
    engine,
)
display(columns)

,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,character_maximum_length,numeric_precision,numeric_scale,is_nullable,column_default
0,public,datastream,1,datastream_id,bigint,int8,NaN,64.0,0.0,NO,nextval('datastream_datastream_id_seq'::regclass)
1,public,datastream,2,datetime_utc,timestamp without time zone,timestamp,NaN,NaN,NaN,NO,NaN
2,public,datastream,3,value,double precision,float8,NaN,53.0,NaN,YES,NaN
3,public,datastream,4,site_id,integer,int4,NaN,32.0,0.0,NO,NaN
4,public,datastream,5,variable_id,integer,int4,NaN,32.0,0.0,NO,NaN
...,...,...,...,...,...,...,...,...,...,...,...
99,public,variable,4,description,character varying,varchar,500.0,NaN,NaN,YES,NaN
100,public,variable,5,variable_type,character varying,varchar,100.0,NaN,NaN,YES,NaN
101,public,variable,6,variable_code,character varying,varchar,100.0,NaN,NaN,YES,NaN
102,public,variable,7,unit_id,integer,int4,NaN,32.0,0.0,YES,NaN


In [6]:
# Compact table-by-table column view.
for table_name, group in columns.groupby("table_name", sort=True):
    print(f"\n{table_name}")
    display(group[["ordinal_position", "column_name", "data_type", "is_nullable", "column_default"]])


datastream


,ordinal_position,column_name,data_type,is_nullable,column_default
0,1,datastream_id,bigint,NO,nextval('datastream_datastream_id_seq'::regclass)
1,2,datetime_utc,timestamp without time zone,NO,NaN
2,3,value,double precision,YES,NaN
3,4,site_id,integer,NO,NaN
4,5,variable_id,integer,NO,NaN
5,6,owner_id,integer,YES,NaN
6,7,qualifier_id,integer,YES,NaN
7,8,processing_level_id,integer,YES,NaN



method


,ordinal_position,column_name,data_type,is_nullable,column_default
8,1,method_id,integer,NO,nextval('method_method_id_seq'::regclass)
9,2,name,character varying,NO,NaN
10,3,description,character varying,YES,NaN
11,4,method_code,character varying,YES,NaN
12,5,method_type,character varying,YES,NaN
13,6,method_link,character varying,YES,NaN
14,7,sensor_manufacturer_name,character varying,YES,NaN
15,8,sensor_model_name,character varying,YES,NaN
16,9,sensor_model_link,character varying,YES,NaN



owner


...

,ordinal_position,column_name,data_type,is_nullable,column_default
38,1,staging_id,bigint,NO,nextval('staging_staging_id_seq'::regclass)
39,2,workspace_name,character varying,YES,NaN
40,3,owner_name,character varying,YES,NaN
41,4,contact_email,character varying,YES,NaN
42,5,site_code,character varying,YES,NaN
43,6,site_name,character varying,YES,NaN
44,7,site_description,character varying,YES,NaN
45,8,sampling_feature_type,character varying,YES,NaN
46,9,site_type,character varying,YES,NaN
47,10,latitude,double precision,YES,NaN



unit


,ordinal_position,column_name,data_type,is_nullable,column_default
91,1,unit_id,integer,NO,nextval('unit_unit_id_seq'::regclass)
92,2,name,character varying,NO,NaN
93,3,symbol,character varying,YES,NaN
94,4,definition,character varying,YES,NaN
95,5,unit_type,character varying,YES,NaN



variable


,ordinal_position,column_name,data_type,is_nullable,column_default
96,1,variable_id,integer,NO,nextval('variable_variable_id_seq'::regclass)
97,2,name,character varying,NO,NaN
98,3,definition,character varying,YES,NaN
99,4,description,character varying,YES,NaN
100,5,variable_type,character varying,YES,NaN
101,6,variable_code,character varying,YES,NaN
102,7,unit_id,integer,YES,NaN
103,8,method_id,integer,YES,NaN


## Variable Names and IDs

In [ ]:
variable_names_and_ids = pd.read_sql_query(
    text("""
    SELECT
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        v.variable_type,
        u.name AS unit_name,
        u.symbol AS unit_symbol
    FROM public.variable v
    LEFT JOIN public.unit u ON v.unit_id = u.unit_id
    ORDER BY v.variable_id
    """),
    engine,
)
display(variable_names_and_ids)

## 5. Primary Keys and Foreign Keys

In [7]:
primary_keys = pd.read_sql_query(
    text("""
    SELECT
        tc.table_schema,
        tc.table_name,
        tc.constraint_name,
        kcu.column_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema = kcu.table_schema
    WHERE tc.constraint_type = 'PRIMARY KEY'
      AND tc.table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY tc.table_schema, tc.table_name, kcu.ordinal_position
    """),
    engine,
)
display(primary_keys)

,table_schema,table_name,constraint_name,column_name
0,public,datastream,datastream_pkey,datastream_id
1,public,method,method_pkey,method_id
2,public,owner,owner_pkey,owner_id
3,public,processing_level,processing_level_pkey,processing_level_id
4,public,qualifier,qualifier_pkey,qualifier_id
5,public,site,site_pkey,site_id
6,public,staging,staging_pkey,staging_id
7,public,unit,unit_pkey,unit_id
8,public,variable,variable_pkey,variable_id


In [8]:
foreign_keys = pd.read_sql_query(
    text("""
    SELECT
        tc.table_schema,
        tc.table_name,
        kcu.column_name,
        ccu.table_schema AS referenced_table_schema,
        ccu.table_name AS referenced_table_name,
        ccu.column_name AS referenced_column_name,
        tc.constraint_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage ccu
        ON ccu.constraint_name = tc.constraint_name
       AND ccu.table_schema = tc.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY tc.table_schema, tc.table_name, kcu.column_name
    """),
    engine,
)
display(foreign_keys)

,table_schema,table_name,column_name,referenced_table_schema,referenced_table_name,referenced_column_name,constraint_name
0,public,datastream,owner_id,public,owner,owner_id,datastream_owner_id_fkey
1,public,datastream,processing_level_id,public,processing_level,processing_level_id,datastream_processing_level_id_fkey
2,public,datastream,qualifier_id,public,qualifier,qualifier_id,datastream_qualifier_id_fkey
3,public,datastream,site_id,public,site,site_id,datastream_site_id_fkey
4,public,datastream,variable_id,public,variable,variable_id,datastream_variable_id_fkey
5,public,variable,method_id,public,method,method_id,variable_method_id_fkey
6,public,variable,unit_id,public,unit,unit_id,variable_unit_id_fkey


## 6. Indexes

In [ ]:
indexes = pd.read_sql_query(
    text("""
    SELECT
        schemaname AS table_schema,
        tablename AS table_name,
        indexname AS index_name,
        indexdef AS index_definition
    FROM pg_indexes
    WHERE schemaname NOT IN ('pg_catalog', 'information_schema')
    ORDER BY schemaname, tablename, indexname
    """),
    engine,
)
display(indexes)

## 7. Row Counts

In [ ]:
inspector = inspect(engine)
prep = engine.dialect.identifier_preparer

row_counts = []
for _, row in tables.iterrows():
    schema = row["table_schema"]
    table = row["table_name"]
    qualified_name = f"{prep.quote_schema(schema)}.{prep.quote(table)}"
    count = pd.read_sql_query(text(f"SELECT COUNT(*) AS row_count FROM {qualified_name}"), engine).iloc[0, 0]
    row_counts.append({"table_schema": schema, "table_name": table, "row_count": count})

row_counts = pd.DataFrame(row_counts).sort_values("row_count", ascending=False)
display(row_counts)

## 8. Sample Rows From Each Table

In [ ]:
for _, row in tables.iterrows():
    schema = row["table_schema"]
    table = row["table_name"]
    qualified_name = f"{prep.quote_schema(schema)}.{prep.quote(table)}"
    print(f"\nSample: {schema}.{table}")
    display(pd.read_sql_query(text(f"SELECT * FROM {qualified_name} LIMIT 5"), engine))

## 9. Datastream Overview

In [9]:
datastream_exists = ((tables["table_schema"] == "public") & (tables["table_name"] == "datastream")).any()

if datastream_exists:
    datastream_overview = pd.read_sql_query(
        text("""
        SELECT
            COUNT(*) AS rows,
            MIN(datetime_utc) AS min_datetime_utc,
            MAX(datetime_utc) AS max_datetime_utc,
            COUNT(*) FILTER (WHERE value IS NULL) AS null_values,
            MIN(value) AS min_value,
            MAX(value) AS max_value,
            AVG(value) AS avg_value
        FROM public.datastream
        """),
        engine,
    )
    display(datastream_overview)
else:
    print("No public.datastream table found. If the database is still in staging mode, inspect public.staging instead.")

,rows,min_datetime_utc,max_datetime_utc,null_values,min_value,max_value,avg_value
0,4067863,2013-10-02 20:15:00,2026-03-24 19:00:00,0,-9999.0,1818.303563,-74.702626


In [10]:
if datastream_exists:
    datastream_by_site_variable = pd.read_sql_query(
        text("""
        SELECT
            s.site_code,
            s.name AS site_name,
            v.variable_code,
            v.name AS variable_name,
            u.symbol AS unit_symbol,
            COUNT(*) AS observation_count,
            MIN(d.datetime_utc) AS first_observation,
            MAX(d.datetime_utc) AS last_observation,
            MIN(d.value) AS min_value,
            MAX(d.value) AS max_value,
            AVG(d.value) AS avg_value
        FROM public.datastream d
        JOIN public.site s ON d.site_id = s.site_id
        JOIN public.variable v ON d.variable_id = v.variable_id
        LEFT JOIN public.unit u ON v.unit_id = u.unit_id
        GROUP BY s.site_code, s.name, v.variable_code, v.name, u.symbol
        ORDER BY observation_count DESC, s.site_code, v.variable_code
        """),
        engine,
    )
    display(datastream_by_site_variable)

,site_code,site_name,variable_code,variable_name,unit_symbol,observation_count,first_observation,last_observation,min_value,max_value,avg_value
0,LR_GC_C,Climate Station at Logan River Golf Course,Precip,Precipitation,cm,838768,2014-03-18 15:30:00,2026-03-24 17:00:00,-9999.00,63.755350,25.367174
1,LR_Mendon_AA,Logan River at Mendon Road (600 South),WaterTemp,Water Temperature,C,436398,2013-10-11 22:15:00,2026-03-24 19:00:00,-9999.00,25.690000,-237.114411
2,LR_WaterLab_AA,Logan River at the Utah Water Research Laboratory west bridge,WaterTemp,Water Temperature,C,435460,2013-10-02 20:15:00,2026-03-24 19:00:00,-9999.00,18.320000,-118.849206
3,LR_WaterLab_AA,Logan River at the Utah Water Research Laboratory west bridge,ODO,"Oxygen, dissolved",mg/L,435148,2013-10-02 20:15:00,2026-03-24 18:00:00,-9999.00,13.180000,-111.457541
4,LR_Mendon_AA,Logan River at Mendon Road (600 South),Discharge,Discharge,cfs,428591,2014-01-01 07:15:00,2026-03-24 19:00:00,-9999.00,1807.356062,-30.607296
5,LR_WaterLab_AA,Logan River at the Utah Water Research Laboratory west bridge,Discharge,Discharge,cfs,428202,2014-01-01 07:00:00,2026-03-24 06:00:00,-9999.00,1818.303563,-97.421942
6,LR_TG_C,Climate Station at Tony Grove,SnowDepth,Snow depth,cm,423331,2014-02-19 21:15:00,2026-03-24 17:00:00,-9999.00,276.700000,-168.283659
7,LR_GC_C,Climate Station at Logan River Golf Course,AirTemp,Air Temperature,C,321544,2017-01-20 23:00:00,2026-03-24 18:00:00,-30.09,80.300000,9.441475
8,LR_TG_C,Climate Station at Tony Grove,AirTemp,Air Temperature,C,320421,2017-02-01 19:45:00,2026-03-24 17:00:00,-28.29,33.860000,5.030111


In [ ]:
if datastream_exists:
    joined_sample = pd.read_sql_query(
        text("""
        SELECT
            d.datastream_id,
            d.datetime_utc,
            d.value,
            s.site_code,
            s.name AS site_name,
            s.latitude,
            s.longitude,
            v.variable_code,
            v.name AS variable_name,
            u.name AS unit_name,
            u.symbol AS unit_symbol,
            m.name AS method_name,
            pl.code AS processing_level_code,
            q.qualifier_code
        FROM public.datastream d
        JOIN public.site s ON d.site_id = s.site_id
        JOIN public.variable v ON d.variable_id = v.variable_id
        LEFT JOIN public.unit u ON v.unit_id = u.unit_id
        LEFT JOIN public.method m ON v.method_id = m.method_id
        LEFT JOIN public.processing_level pl ON d.processing_level_id = pl.processing_level_id
        LEFT JOIN public.qualifier q ON d.qualifier_id = q.qualifier_id
        ORDER BY d.datetime_utc
        LIMIT 20
        """),
        engine,
    )
    display(joined_sample)

## 10. Missingness by Table

In [ ]:
def null_summary(schema, table):
    table_columns = columns.loc[
        (columns["table_schema"] == schema) & (columns["table_name"] == table),
        "column_name",
    ].tolist()
    qualified_name = f"{prep.quote_schema(schema)}.{prep.quote(table)}"
    expressions = [f"COUNT(*) AS total_rows"]
    expressions += [f"COUNT(*) FILTER (WHERE {prep.quote(col)} IS NULL) AS {prep.quote(col + '_nulls')}" for col in table_columns]
    query = f"SELECT {', '.join(expressions)} FROM {qualified_name}"
    return pd.read_sql_query(text(query), engine)

# Change this list if you want to profile fewer or more tables.
tables_to_profile = row_counts.head(10)[["table_schema", "table_name"]]

for _, row in tables_to_profile.iterrows():
    print(f"\nNull summary: {row['table_schema']}.{row['table_name']}")
    display(null_summary(row["table_schema"], row["table_name"]))

## 11. Useful Starting Queries

In [12]:
# Sites
pd.read_sql_query(
    text("""
    SELECT site_id, site_code, name, site_type, latitude, longitude, elevation_m, state, county
    FROM public.site
    ORDER BY site_code
    """),
    engine,
)

,site_id,site_code,name,site_type,latitude,longitude,elevation_m,state,county
0,2,LR_GC_C,Climate Station at Logan River Golf Course,Atmosphere,41.705643,-111.854268,1364.89,Utah,Cache
1,4,LR_Mendon_AA,Logan River at Mendon Road (600 South),Stream,41.720533,-111.886928,1353.50,Utah,Cache
2,1,LR_TG_C,Climate Station at Tony Grove,Atmosphere,41.885493,-111.568767,1927.86,Utah,Cache
3,3,LR_WaterLab_AA,Logan River at the Utah Water Research Laboratory west bridge,Stream,41.739034,-111.795742,1414.00,Utah,Cache


In [13]:
# Variables with units and methods
pd.read_sql_query(
    text("""
    SELECT
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        v.variable_type,
        u.name AS unit_name,
        u.symbol AS unit_symbol,
        m.name AS method_name
    FROM public.variable v
    LEFT JOIN public.unit u ON v.unit_id = u.unit_id
    LEFT JOIN public.method m ON v.method_id = m.method_id
    ORDER BY v.variable_code, v.name
    """),
    engine,
)

,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol,method_name
0,5,AirTemp,Air Temperature,Climate,degree Celsius,C,E+E Elektronik EE08 temperature and relative humidity sensor
1,4,Discharge,Discharge,Hydrology,cubic feet per second,cfs,Stage-Discharge Rating Curve
2,3,ODO,"Oxygen, dissolved",Water Quality,milligrams per liter,mg/L,YSI EXO multi-parameter water quality sonde dissolved oxygen sensor
3,6,Precip,Precipitation,Hydrology,centimeter,cm,Geonor T-200B rain gage
4,1,SnowDepth,Snow depth,Hydrology,centimeter,cm,Judd Snow Sensor
5,2,WaterTemp,Water Temperature,Water Quality,degree Celsius,C,YSI EXO multi-parameter water quality sonde temperature and specific conductance sensor


In [11]:
# Daily summary by site and variable
pd.read_sql_query(
    text("""
    SELECT
        DATE_TRUNC('day', d.datetime_utc) AS day_utc,
        s.site_code,
        v.variable_code,
        COUNT(*) AS n_observations,
        AVG(d.value) AS avg_value,
        MIN(d.value) AS min_value,
        MAX(d.value) AS max_value
    FROM public.datastream d
    JOIN public.site s ON d.site_id = s.site_id
    JOIN public.variable v ON d.variable_id = v.variable_id
    GROUP BY day_utc, s.site_code, v.variable_code
    ORDER BY day_utc, s.site_code, v.variable_code
    LIMIT 100
    """),
    engine,
)

,day_utc,site_code,variable_code,n_observations,avg_value,min_value,max_value
0,2013-10-02,LR_WaterLab_AA,ODO,15,10.018667,9.95,10.10
1,2013-10-02,LR_WaterLab_AA,WaterTemp,15,9.434667,9.38,9.48
2,2013-10-03,LR_WaterLab_AA,ODO,96,9.882812,9.50,10.23
3,2013-10-03,LR_WaterLab_AA,WaterTemp,96,9.122500,7.64,9.53
4,2013-10-04,LR_WaterLab_AA,ODO,96,10.254792,10.07,10.70
...,...,...,...,...,...,...,...
95,2013-11-05,LR_WaterLab_AA,WaterTemp,96,4.576667,4.30,4.80
96,2013-11-06,LR_Mendon_AA,WaterTemp,96,5.787500,5.18,6.90
97,2013-11-06,LR_WaterLab_AA,ODO,96,11.375938,11.17,11.77
98,2013-11-06,LR_WaterLab_AA,WaterTemp,96,4.561354,4.40,4.95
